# Amharic response-level distillation: base run, LoRA training, distilled run, comparison

Run the cells **top to bottom** on a free Colab GPU (Runtime > Change runtime type > **T4 GPU**).
Everything (base run, training, distilled run) happens in **one session on the same GPU**, as the task requires.

**You need these 3 files** (upload when asked, or put them in `MyDrive/amharic_distill_out/`):
1. `train_gemini_gemma-4-26b-a4b-it.csv` (your final training data: prompt, teacher_response, task, source)
2. `eval_prompts_60.csv`
3. `predictions_template.csv` (or `predictions_with_teacher.csv` if you filled `teacher_output`)

Student: `google/gemma-3-270m-it` (you must accept its licence on Hugging Face and give Colab a token).
Expected time on a T4: base run ~5-10 min, training ~30-60 min, distilled run ~10-15 min.

In [5]:
# 0. GPU + packages
!nvidia-smi
!pip install -q peft sacrebleu accelerate

Wed Oct  7 00:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# 1. Configuration, seed, versions
import os, sys, json, csv, glob, math, time, random, gc, platform, subprocess, shutil, re, unicodedata, hashlib
from collections import Counter
import numpy as np
import torch

SEED = 42
STUDENT = "google/gemma-3-270m-it"
MY_NAME = "simachew"            # your predictions file will be <MY_NAME>.csv (change to your name)

MAX_NEW_TOKENS = 256            # evaluation settings fixed by the task: greedy, 256 new tokens, batch size 1
MAX_LEN = 1024                  # training sequence length (longer pairs are skipped, never truncated)
EPOCHS, LR = 3, 2e-4
MICRO_BATCH, ACCUM = 2, 8       # effective batch size 16 (if you get an out-of-memory error: MICRO_BATCH=1, ACCUM=16)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
VAL_SIZE = 100                  # teacher answers held out (not trained on) to track validation loss
RUN_MERGED_TIMING = True        # also time the LoRA-merged model (speed/memory cost of the adapter vs merged)
USE_DRIVE = True                # save everything to Google Drive so a disconnect does not lose work
REQUIRE_GPU = True
CONTENT_DIR = "/content"

OUT = os.path.join(CONTENT_DIR, "amharic_distill_out")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/amharic_distill_out"
    except Exception as e:
        print("Google Drive not mounted, using local folder:", e)
os.makedirs(OUT, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if REQUIRE_GPU:
    assert DEVICE == "cuda", "No GPU: Runtime > Change runtime type > T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# bf16 needs compute capability >= 8 (A100/L4); the free T4 uses fp32 (safe for Gemma-3, a bit slower)
DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float32

import transformers, peft
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "cuda": torch.version.cuda, "gpu": GPU_NAME, "dtype": str(DTYPE), "seed": SEED}
json.dump(VERSIONS, open(f"{OUT}/run_info.json", "w"), indent=2)
open(f"{OUT}/requirements_colab.txt", "w").write(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(VERSIONS)
print("outputs ->", OUT)

Mounted at /content/drive
{'python': '3.13.15', 'torch': '2.11.0+cu130', 'transformers': '5.18.0', 'peft': '0.21.1', 'cuda': '13.0', 'gpu': 'Tesla T4', 'dtype': 'torch.float32', 'seed': 42}
outputs -> /content/drive/MyDrive/amharic_distill_out


In [7]:
# 2. Hugging Face login (Gemma is a gated model: accept the licence on its page first)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))          # Colab: key icon on the left > add secret HF_TOKEN
except Exception as e:
    print("No HF_TOKEN secret found (", type(e).__name__, ") -> paste a token below")
    from huggingface_hub import notebook_login
    notebook_login()

In [8]:
# 3. Locate / upload the input files and load them
def find(patterns, exclude=()):
    for base in (OUT, CONTENT_DIR):
        for pat in patterns:
            for h in sorted(glob.glob(os.path.join(base, pat))):
                if not any(x in os.path.basename(h) for x in exclude):
                    return h
    return None

def locate():
    return (find(["train_*.csv"], exclude=("_full",)), find(["eval_prompts*.csv"]),
            find(["*with_teacher*.csv", "predictions_template*.csv"]))

TRAIN_CSV, EVAL_CSV, PRED_CSV = locate()
if not (TRAIN_CSV and EVAL_CSV and PRED_CSV):
    from google.colab import files
    print("Upload: train_*.csv, eval_prompts_60.csv and predictions_template.csv")
    files.upload()
    TRAIN_CSV, EVAL_CSV, PRED_CSV = locate()
print("train:", TRAIN_CSV, "\neval :", EVAL_CSV, "\npred :", PRED_CSV)

def read_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

train_rows, eval_rows, pred_rows = read_csv(TRAIN_CSV), read_csv(EVAL_CSV), read_csv(PRED_CSV)
assert {"prompt", "teacher_response", "task", "source"} <= set(train_rows[0]), list(train_rows[0])
assert len(eval_rows) == 60, len(eval_rows)
assert [r["eval_id"] for r in pred_rows] == [r["eval_id"] for r in eval_rows], "template ids differ from eval ids"
print(len(train_rows), "training pairs |", dict(Counter(r["task"] for r in train_rows)))

# --- isolation check: no eval prompt (or near copy) may be in the training data ---
def norm(t):
    t = unicodedata.normalize("NFC", t).casefold()
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()
def sha(t): return hashlib.sha256(norm(t).replace(" ", "").encode("utf-8")).hexdigest()
def grams(t, n=4):
    t = norm(t); return {t[i:i + n] for i in range(max(1, len(t) - n + 1))}
def jaccard(a, b): return len(a & b) / len(a | b) if a and b else 0.0
def contain(a, b): return len(a & b) / len(a) if a else 0.0

ev_prompts = [r["prompt"] for r in eval_rows]
ev_g, ev_h = [grams(p) for p in ev_prompts], {sha(p) for p in ev_prompts}
leaks = 0
for r in train_rows:
    p = r["prompt"]
    if sha(p) in ev_h: leaks += 1; continue
    pg = grams(p)
    if any(jaccard(pg, e) > 0.4 or contain(e, pg) > 0.6 for e in ev_g): leaks += 1
print("LEAKS vs the 60 eval prompts:", leaks, "(must be 0)")
if leaks: print("WARNING: remove these prompts from the training data before training!")

train: /content/drive/MyDrive/amharic_distill_out/train_gemini_gemma-4-26b-a4b-it.csv 
eval : /content/drive/MyDrive/amharic_distill_out/eval_prompts_60.csv 
pred : /content/drive/MyDrive/amharic_distill_out/predictions_template.csv
1934 training pairs | {'closed_book_knowledge': 451, 'instruction_following': 500, 'translation': 487, 'summarization': 496}
LEAKS vs the 60 eval prompts: 0 (must be 0)


In [9]:
# 4. Model/tokenizer helpers, generation and evaluation loop
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(STUDENT)

def load_base():
    kw = dict(attn_implementation="eager")        # recommended for Gemma-3 (training and inference)
    try:
        m = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=DTYPE, **kw)
    except Exception:
        m = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=DTYPE, **kw)
    return m.to(DEVICE).eval()

def free_gpu():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()
def reset_peak():
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
def peak_mb():
    return torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else 0.0
def dir_mb(path, suffixes=None):
    tot = 0
    for root, _, fs in os.walk(path):
        for f in fs:
            if suffixes is None or f.endswith(suffixes): tot += os.path.getsize(os.path.join(root, f))
    return tot / 2**20

def prompt_text(prompt):
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate_one(model, prompt):
    enc = tok(prompt_text(prompt), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    sync(); t0 = time.perf_counter()
    out = model.generate(**enc, do_sample=False, temperature=None, top_p=None, top_k=None,
                         max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
    sync(); dt = time.perf_counter() - t0
    new_ids = out[0, enc["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True), dt, int(new_ids.shape[0])   # raw text, seconds, new tokens

def run_eval(model, label):
    # greedy decoding, batch size 1, max_new_tokens=256, same GPU; 2 untimed warm-up generations first
    for r in eval_rows[:2]: generate_one(model, r["prompt"])
    free_gpu(); reset_peak()
    res = []
    for r in tqdm(eval_rows, desc=label):
        text, dt, n = generate_one(model, r["prompt"])
        res.append({"eval_id": r["eval_id"], "task": r["task"], "direction": r.get("direction", ""),
                    "text": text, "seconds": dt, "new_tokens": n})
    secs = sum(x["seconds"] for x in res); toks = sum(x["new_tokens"] for x in res)
    stats = {"label": label, "sec_per_answer_mean": secs / len(res),
             "sec_per_answer_median": float(np.median([x["seconds"] for x in res])),
             "tokens_per_sec": toks / secs, "total_new_tokens": toks, "peak_mem_mb": peak_mb(),
             "n_hit_256": sum(x["new_tokens"] >= MAX_NEW_TOKENS for x in res)}
    print(json.dumps(stats, indent=1))
    return res, stats

def show(res, k=1):
    seen = Counter()
    for x in res:
        if seen[x["task"]] < k:
            seen[x["task"]] += 1
            print("-", x["eval_id"], x["task"], "|", f"{x['seconds']:.1f}s", x["new_tokens"], "tok\n ", x["text"][:300].replace("\n", " "), "\n")

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

In [10]:
# 5. BASE RUN (original student, no fine-tuning)
free_gpu()
model = load_base()
base_total_params = sum(p.numel() for p in model.parameters())
base_dir = STUDENT if os.path.isdir(STUDENT) else __import__("huggingface_hub").snapshot_download(STUDENT)
base_disk_mb = dir_mb(base_dir, (".safetensors",))
base_results, base_stats = run_eval(model, "base")
json.dump({"results": base_results, "stats": base_stats}, open(f"{OUT}/base_outputs.json", "w"), ensure_ascii=False, indent=1)
print(f"parameters: {base_total_params:,} | checkpoint on disk: {base_disk_mb:.0f} MB")
show(base_results)
del model; free_gpu()

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

base:   0%|          | 0/60 [00:00<?, ?it/s]

{
 "label": "base",
 "sec_per_answer_mean": 3.994079393816666,
 "sec_per_answer_median": 2.0234530850000283,
 "tokens_per_sec": 25.67967647950431,
 "total_new_tokens": 6154,
 "peak_mem_mb": 1158.421875,
 "n_hit_256": 17
}
parameters: 268,098,176 | checkpoint on disk: 511 MB
- E01 closed_book_knowledge | 5.3s 40 tok
  በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት በአ satu ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው.  

- E16 instruction_following | 1.0s 27 tok
  በኢትዮጵያዊ ወizards ተሽከርካሪ ανাসாகለው ክበ dignity 40. 

- E31 translation | 1.8s 62 tok
  உண்மைதான்! "ከ አማርኛ ወደ English ተርጉም" (Kě yī móxíng yào English) என்பதன் అర్థం হল, "આपर्यंत જ पदचिह्न (plaque) ની જરૂર છે" (Yàfǐng jī yào English).  

- E46 summarization | 8.3s 256 tok
  በፌስቡክ ገጹ ላይ በኬንያ መሪዎች ስር ከመተዳደር ጎንደር ውስጥ የቤት ውስጥ አገልጋይ ሆኖ መኖርን እንደሚመርጥ የሚናገረው ብሪያን ኪቤት ቤራ ሰኞ እ nephews ቤተ መንግሥቱ ውስጥ ለመግባት የሞከረው። በጆሞ ኬንያታ የግብርናና የቴክኖሎጂ ዩኒቨርስቲ ውስጥ የአምስተኛ ዓመት የሜካኒካል ኢንጂነሪንግ ተማሪ የሆነው ብራያን፤ ቤተ መንግሥት ሊገባ ሲል ግራ ትከሻው ላይ በጥይት ከቆሰለ በኋላ ሆስፒ colleges ገብቷል። •እምቦጭ የጣና ቂርቆስ መነኮሳትን ከገዳሙ እያስለቀቀ ነ

In [11]:
# 6. Prepare training data: student's own chat template, loss ONLY on the answer tokens
def enc_pair(prompt, answer):
    msgs = [{"role": "user", "content": prompt}]
    p_text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    f_text = tok.apply_chat_template(msgs + [{"role": "assistant", "content": answer}], tokenize=False)
    p_ids = tok(p_text, add_special_tokens=False)["input_ids"]
    f_ids = tok(f_text, add_special_tokens=False)["input_ids"]
    n = 0                                            # common prefix (robust to merges at the prompt/answer boundary)
    while n < min(len(p_ids), len(f_ids)) and p_ids[n] == f_ids[n]: n += 1
    return f_ids, [-100] * n + f_ids[n:], n != len(p_ids)

rng = random.Random(SEED)
rows = train_rows[:]; rng.shuffle(rows)
val_rows, tr_rows = rows[:VAL_SIZE], rows[VAL_SIZE:]
def encode_all(rs):
    data, too_long, boundary = [], 0, 0
    for r in rs:
        ids, lab, b = enc_pair(r["prompt"], r["teacher_response"])
        if len(ids) > MAX_LEN: too_long += 1; continue
        boundary += b; data.append((ids, lab))
    return data, too_long, boundary
train_data, tl1, b1 = encode_all(tr_rows)
val_data, tl2, b2 = encode_all(val_rows)
lens = [len(x[0]) for x in train_data]
print(f"train {len(train_data)} | val {len(val_data)} | skipped (>{MAX_LEN} tokens): {tl1 + tl2} | boundary mismatches: {b1 + b2}")
print("sequence length median/p95/max:", int(np.median(lens)), int(np.percentile(lens, 95)), max(lens))
ids, lab, _ = enc_pair(tr_rows[0]["prompt"], tr_rows[0]["teacher_response"])
print("\nMASKED (prompt) tail :", repr(tok.decode([i for i, l in zip(ids, lab) if l == -100])[-150:]))
print("TRAINED (answer) head:", repr(tok.decode([i for i, l in zip(ids, lab) if l != -100])[:150]))

train 1834 | val 100 | skipped (>1024 tokens): 0 | boundary mismatches: 0
sequence length median/p95/max: 255 750 1010

MASKED (prompt) tail : 'ጠፍ የ፪ ቢሊዮን ብር በጀት መመደቡንም በዜናው ላይ ተጠቅሷል። ይህ እርምጃ በገበያው ላይ ያለውን የዋጋ መናር ለመቀነስና ለገበሬዎች የተሻለ ገቢ ለማስገኘት ይረዳናል ተብሎ ይታመናል።<end_of_turn>\n<start_of_turn>model\n'
TRAINED (answer) head: 'በሰላም ተራራ አካባቢ የትራንስፖርት ወጪ መጨመር የግብርና ምርቶችና የጥራጥሬ ዋጋ እንዲናር ምክንያት ሆኗል። መንግሥት የዋጋ ንረቱን ለመቀነስና የምርት ፍሰቱን ለማቀላጠፍ የ2 ቢሊዮን ብር በጀት መድቧል።<end_of_turn>\n'


In [12]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
# 7. LoRA fine-tuning (custom loop: no extra libraries, fully reproducible with the seed)
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import get_cosine_schedule_with_warmup

ADAPTER_DIR, TRAIN_INFO = f"{OUT}/adapter_final", f"{OUT}/training_info.json"

def collate(exs):
    L = max(len(x[0]) for x in exs)
    ids = torch.full((len(exs), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(exs), L), -100, dtype=torch.long)
    att = torch.zeros((len(exs), L), dtype=torch.long)
    for i, (f, l) in enumerate(exs):
        n = len(f); ids[i, :n] = torch.tensor(f); lab[i, :n] = torch.tensor(l); att[i, :n] = 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": att.to(DEVICE), "labels": lab.to(DEVICE)}

def make_batches(data, bs, shuffle, rng):
    idx = list(range(len(data)))
    if shuffle:                                   # length-grouped shuffling: less padding, faster steps
        rng.shuffle(idx)
        chunks = [idx[i:i + bs * 16] for i in range(0, len(idx), bs * 16)]
        idx = [j for c in chunks for j in sorted(c, key=lambda k: len(data[k][0]))]
    batches = [idx[i:i + bs] for i in range(0, len(idx), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

@torch.no_grad()
def val_loss(m, data, bs=4):
    was_training = m.training; m.eval(); tot, cnt = 0.0, 0
    for b in make_batches(data, bs, False, None):
        batch = collate([data[k] for k in b])
        n = int((batch["labels"][:, 1:] != -100).sum())
        tot += m(**batch).loss.item() * n; cnt += n
    if was_training: m.train()
    return tot / cnt

if os.path.exists(ADAPTER_DIR) and os.path.exists(TRAIN_INFO):
    train_info = json.load(open(TRAIN_INFO))
    print("Adapter already trained -> skipping training:\n", json.dumps({k: v for k, v in train_info.items() if k != "history"}, indent=1))
else:
    free_gpu(); set_seed(SEED)
    model = load_base()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                                             task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable LoRA parameters: {n_train:,} of {n_all:,} ({100 * n_train / n_all:.2f}%)")

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(make_batches(train_data, MICRO_BATCH, False, None)) / ACCUM)
    total_steps = steps_per_epoch * EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.03 * total_steps)), total_steps)
    OOM = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

    def fwd_bwd(exs, scale):
        try:
            loss = model(**collate(exs)).loss
            (loss * scale).backward()
            return loss.item()
        except OOM:
            if len(exs) == 1: raise
            free_gpu(); mid = len(exs) // 2
            return (fwd_bwd(exs[:mid], scale / 2) + fwd_bwd(exs[mid:], scale / 2)) / 2

    val_hist = [("before_training", val_loss(model, val_data))]
    print("validation loss before training:", round(val_hist[0][1], 4))
    history, step = [], 0
    rng_tr = random.Random(SEED)
    reset_peak(); t_start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        batches = make_batches(train_data, MICRO_BATCH, True, rng_tr)
        run, cnt = 0.0, 0
        for bi, b in enumerate(batches):
            run += fwd_bwd([train_data[k] for k in b], 1.0 / ACCUM); cnt += 1
            if (bi + 1) % ACCUM == 0 or bi == len(batches) - 1:
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if step % 10 == 0 or step == total_steps:
                    el = time.perf_counter() - t_start
                    history.append({"step": step, "loss": run / cnt})
                    print(f"epoch {epoch} step {step}/{total_steps} loss {run / cnt:.4f} | {el / 60:.1f} min elapsed, ETA {el / step * (total_steps - step) / 60:.1f} min")
                    run, cnt = 0.0, 0
        v = val_loss(model, val_data); val_hist.append((f"epoch_{epoch}", v))
        print(f"== epoch {epoch} validation loss: {v:.4f}")
        model.save_pretrained(f"{OUT}/adapter_epoch{epoch}")          # safe copy after every epoch
    train_seconds, train_peak = time.perf_counter() - t_start, peak_mb()
    model.save_pretrained(ADAPTER_DIR)
    train_info = {"train_seconds": train_seconds, "train_minutes": train_seconds / 60, "peak_train_mem_mb": train_peak,
                  "trainable_params": n_train, "all_params_with_adapter": n_all, "steps": total_steps,
                  "train_examples": len(train_data), "val_examples": len(val_data), "val_loss": val_hist,
                  "epochs": EPOCHS, "lr": LR, "effective_batch": MICRO_BATCH * ACCUM, "max_len": MAX_LEN,
                  "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT, "gpu": GPU_NAME,
                  "dtype": str(DTYPE), "seed": SEED, "history": history}
    json.dump(train_info, open(TRAIN_INFO, "w"), indent=1)
    print(f"training done in {train_seconds / 60:.1f} min | peak memory {train_peak:.0f} MB | adapter -> {ADAPTER_DIR}")
    try:
        import matplotlib.pyplot as plt
        plt.plot([h["step"] for h in history], [h["loss"] for h in history]); plt.xlabel("step"); plt.ylabel("train loss")
        plt.title("LoRA training loss"); plt.savefig(f"{OUT}/train_loss.png", dpi=120); plt.show()
    except Exception as e:
        print("plot skipped:", e)
    del model, opt; free_gpu()

Adapter already trained -> skipping training:
 {
 "train_seconds": 1538.4207193510001,
 "train_minutes": 25.64034532251667,
 "peak_train_mem_mb": 9626.44970703125,
 "trainable_params": 3796992,
 "all_params_with_adapter": 271895168,
 "steps": 345,
 "train_examples": 1834,
 "val_examples": 100,
 "val_loss": [
  [
   "before_training",
   3.7382588236812033
  ],
  [
   "epoch_1",
   1.6312033807539696
  ],
  [
   "epoch_2",
   1.5175348400825526
  ],
  [
   "epoch_3",
   1.5055592697390925
  ]
 ],
 "epochs": 3,
 "lr": 0.0002,
 "effective_batch": 16,
 "max_len": 1024,
 "lora_r": 16,
 "lora_alpha": 32,
 "lora_dropout": 0.05,
 "gpu": "Tesla T4",
 "dtype": "torch.float32",
 "seed": 42
}


In [14]:
# 8. DISTILLED RUN (original student + LoRA adapter, same GPU, same settings)
free_gpu()
student = load_base()
dist_model = PeftModel.from_pretrained(student, ADAPTER_DIR).eval()
dist_results, dist_stats = run_eval(dist_model, "distilled (adapter loaded)")
json.dump({"results": dist_results, "stats": dist_stats}, open(f"{OUT}/distilled_outputs.json", "w"), ensure_ascii=False, indent=1)
show(dist_results)

merged_stats = None
if RUN_MERGED_TIMING:      # speed/memory with the adapter merged into the weights (what you would deploy)
    merged = dist_model.merge_and_unload().eval()
    merged_results, merged_stats = run_eval(merged, "distilled (merged)")
    same = sum(a["text"] == b["text"] for a, b in zip(dist_results, merged_results))
    print(f"merged model gives the identical text for {same}/60 prompts (tiny numeric differences are normal)")
    json.dump(merged_stats, open(f"{OUT}/merged_stats.json", "w"), indent=1)
del dist_model, student
free_gpu()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

distilled (adapter loaded):   0%|          | 0/60 [00:00<?, ?it/s]

{
 "label": "distilled (adapter loaded)",
 "sec_per_answer_mean": 3.3546310135166624,
 "sec_per_answer_median": 2.1832997825000575,
 "tokens_per_sec": 21.050501942578524,
 "total_new_tokens": 4237,
 "peak_mem_mb": 1181.28369140625,
 "n_hit_256": 1
}
- E01 closed_book_knowledge | 2.4s 54 tok
  በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት የነፃነት ባለቤት ሊሆኑ የቻሉት የነፃነት ባለቤት አምጪው አምጪ አምጪ (ኢትዮጵያ) ነው። 

- E16 instruction_following | 1.3s 28 tok
  የተሽከርካሪው እስከ 40 ኪሎ ሜትር መጓዝ ትችላለች ተብሏል። 

- E31 translation | 0.5s 9 tok
  The question you ask is very important. 

- E46 summarization | 4.8s 105 tok
  በፌስቡክ ገጹ ላይ በኬንያ መሪዎች ስር ከመተዳደር ጎንደር ውስጥ የቤት ውስጥ አገልጋይ ሆኖ መኖርን እንደሚመርጥ የሚናገረው ብራያን ቤተ መንግሥቱ ሊገባ ሲሆን፣ በጆሞ ኬንያታ የግብርናና የቴክኖሎጂ ዩኒቨርስቲ ውስጥ የአምስተኛ ዓመት የሜካኒካል ኢንጂነሪንግ ተማሪ የሆነው ብራያን ነው። 



distilled (merged):   0%|          | 0/60 [00:00<?, ?it/s]

{
 "label": "distilled (merged)",
 "sec_per_answer_mean": 2.154950836666664,
 "sec_per_answer_median": 1.6911304464998693,
 "tokens_per_sec": 32.76950242442581,
 "total_new_tokens": 4237,
 "peak_mem_mb": 1158.17041015625,
 "n_hit_256": 1
}
merged model gives the identical text for 60/60 prompts (tiny numeric differences are normal)


In [15]:
# 9. Fill the predictions file (raw, unedited generations) -> <MY_NAME>.csv
b = {x["eval_id"]: x for x in base_results}
d = {x["eval_id"]: x for x in dist_results}
fields = list(pred_rows[0].keys())
missing = [k for k in ("base_output", "distilled_output") if k not in fields]
assert not missing, f"template lacks required columns: {missing}"
for r in pred_rows:
    i = r["eval_id"]
    vals = {"base_output": b[i]["text"], "distilled_output": d[i]["text"],
            "base_seconds": round(b[i]["seconds"], 3), "base_new_tokens": b[i]["new_tokens"],
            "distilled_seconds": round(d[i]["seconds"], 3), "distilled_new_tokens": d[i]["new_tokens"]}
    for k, v in vals.items():
        if k in fields:                      # timing columns are optional: fill only if the template has them
            r[k] = v
PRED_OUT = f"{OUT}/{MY_NAME}.csv"
with open(PRED_OUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(pred_rows)
filled = sum(1 for r in pred_rows if r["base_output"] != "" and r["distilled_output"] != "")
print(f"{PRED_OUT}: {len(pred_rows)} rows, both outputs filled for {filled}")
print("teacher_output filled:", sum(1 for r in pred_rows if r.get("teacher_output")), "(optional column)")

/content/drive/MyDrive/amharic_distill_out/simachew.csv: 60 rows, both outputs filled for 60
teacher_output filled: 0 (optional column)


In [16]:
# 10. Compare base vs distilled (no reference answers exist, so we report transparent proxy metrics)
import pandas as pd
from sacrebleu.metrics import CHRF

def fidel_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum("\u1200" <= c <= "\u137F" for c in letters) / len(letters) if letters else 0.0
def script_ok(t, direction):                     # translation am->en must be English, everything else Amharic
    fr = fidel_ratio(t)
    if direction == "am->en": return fr <= 0.1 and any(c.isascii() and c.isalpha() for c in t)
    return fr >= 0.8
def looping(t):                                   # >30% of word trigrams are repeats
    w = t.split()
    if len(w) < 12: return False
    tri = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 1 - len(set(tri)) / len(tri) > 0.3

teacher = {r["eval_id"]: r.get("teacher_output", "") for r in pred_rows}
have_teacher = any(teacher.values())
chrf = CHRF(word_order=2)
rows_out = []
for task in ["closed_book_knowledge", "instruction_following", "translation", "summarization"]:
    for name, res in (("base", base_results), ("distilled", dist_results)):
        xs = [x for x in res if x["task"] == task]
        row = {"task": task, "model": name, "n": len(xs),
               "right_script_%": 100 * np.mean([script_ok(x["text"], x["direction"]) for x in xs]),
               "empty_%": 100 * np.mean([not x["text"].strip() for x in xs]),
               "looping_%": 100 * np.mean([looping(x["text"]) for x in xs]),
               "hit_256_%": 100 * np.mean([x["new_tokens"] >= MAX_NEW_TOKENS for x in xs]),
               "mean_new_tokens": np.mean([x["new_tokens"] for x in xs])}
        if have_teacher:
            pairs = [(x["text"], teacher[x["eval_id"]]) for x in xs if teacher[x["eval_id"]]]
            row["chrF++_vs_teacher"] = chrf.corpus_score([p[0] for p in pairs], [[p[1] for p in pairs]]).score if pairs else float("nan")
        rows_out.append(row)
cmp_df = pd.DataFrame(rows_out).round(1)
cmp_df.to_csv(f"{OUT}/comparison_by_task.csv", index=False)
display(cmp_df)
if not have_teacher:
    print("(teacher_output is empty: run `answer-eval` locally and upload predictions_with_teacher.csv to add chrF++ vs the teacher)")
print("\nvalidation loss on held-out teacher answers:", [(k, round(v, 4)) for k, v in train_info["val_loss"]])

print("\n==== side-by-side examples (2 per task) ====")
for task in ["closed_book_knowledge", "instruction_following", "translation", "summarization"]:
    for i in [x["eval_id"] for x in base_results if x["task"] == task][:2]:
        print(f"\n[{i}] {task}\nPROMPT : {next(r['prompt'] for r in eval_rows if r['eval_id'] == i)[-220:]!r}")
        print("BASE   :", b[i]["text"][:260].replace("\n", " "))
        print("DISTIL :", d[i]["text"][:260].replace("\n", " "))

,task,model,n,right_script_%,empty_%,looping_%,hit_256_%,mean_new_tokens
0,closed_book_knowledge,base,15,26.7,0.0,33.3,33.3,112.9
1,closed_book_knowledge,distilled,15,93.3,0.0,20.0,6.7,101.3
2,instruction_following,base,15,33.3,0.0,13.3,26.7,92.9
3,instruction_following,distilled,15,100.0,0.0,0.0,0.0,44.3
4,translation,base,15,6.7,0.0,0.0,0.0,31.5
5,translation,distilled,15,100.0,0.0,0.0,0.0,26.9
6,summarization,base,15,53.3,0.0,6.7,53.3,172.9
7,summarization,distilled,15,100.0,0.0,0.0,0.0,110.0


(teacher_output is empty: run `answer-eval` locally and upload predictions_with_teacher.csv to add chrF++ vs the teacher)

validation loss on held-out teacher answers: [('before_training', 3.7383), ('epoch_1', 1.6312), ('epoch_2', 1.5175), ('epoch_3', 1.5056)]

==== side-by-side examples (2 per task) ====

[E01] closed_book_knowledge
PROMPT : 'በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት በአንድ ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው?'
BASE   : በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት በአ satu ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው. 
DISTIL : በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት የነፃነት ባለቤት ሊሆኑ የቻሉት የነፃነት ባለቤት አምጪው አምጪ አምጪ (ኢትዮጵያ) ነው።

[E02] closed_book_knowledge
PROMPT : 'የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበትን ተፈጥሯዊ ሂደት አብራራ።'
BASE   : የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበትን ተፈጥሯዊ ሂደት አብራራ.
DISTIL : የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበትን ተፈጥሯዊ ሂደት አብራራ ነው። ይህ ሂደት በዘርናዊው የጤና ሥርዓት ውስጥ ያለውን የጤና እና የውሃ መከላከያን ለማረጋገጥ ይረዳል።

[E16] instruction_following
PROMPT : ' በአንድ ሊትር እስከ 40 ኪሎ ሜትር መጓዝ ትችላለች ተብሏል፡፡ ታዳጊው የ11ኛ ክፍል ተማሪ ሲሆን ከዚህ በፊ

In [17]:
# 11. Efficiency table (for the README) + save everything
from huggingface_hub import snapshot_download
adapter_mb = dir_mb(ADAPTER_DIR)
adapter_params = train_info["trainable_params"]
def row(label, base, dist, merged=None, fmt="{:.2f}"):
    f = lambda v: "-" if v is None else fmt.format(v)
    return f"| {label} | {f(base)} | {f(dist)} | {f(merged)} |"
ms = merged_stats
lines = [
    f"GPU: **{GPU_NAME}** | dtype {DTYPE} | greedy decoding, max_new_tokens={MAX_NEW_TOKENS}, batch size 1 | seed {SEED}",
    "",
    "| metric | base | distilled (adapter loaded) | distilled (merged) |",
    "|---|---|---|---|",
    row("parameters (total)", base_total_params, base_total_params + adapter_params, base_total_params, "{:,.0f}"),
    row("trainable parameters (LoRA)", 0, adapter_params, 0, "{:,.0f}"),
    row("size on disk, full model (MB)", base_disk_mb, base_disk_mb, base_disk_mb, "{:.0f}"),
    row("adapter size on disk (MB)", None, adapter_mb, None),
    row("peak inference memory (MB)", base_stats["peak_mem_mb"], dist_stats["peak_mem_mb"], ms["peak_mem_mb"] if ms else None, "{:.0f}"),
    row("seconds per answer (mean)", base_stats["sec_per_answer_mean"], dist_stats["sec_per_answer_mean"], ms["sec_per_answer_mean"] if ms else None),
    row("seconds per answer (median)", base_stats["sec_per_answer_median"], dist_stats["sec_per_answer_median"], ms["sec_per_answer_median"] if ms else None),
    row("tokens per second", base_stats["tokens_per_sec"], dist_stats["tokens_per_sec"], ms["tokens_per_sec"] if ms else None, "{:.1f}"),
    row("answers that hit 256 tokens", base_stats["n_hit_256"], dist_stats["n_hit_256"], ms["n_hit_256"] if ms else None, "{:.0f}"),
    "",
    f"Training: {train_info['train_minutes']:.1f} min on {train_info['gpu']} | {train_info['epochs']} epochs, {train_info['steps']} steps, "
    f"effective batch {train_info['effective_batch']}, lr {train_info['lr']}, LoRA r={train_info['lora_r']} alpha={train_info['lora_alpha']} "
    f"dropout={train_info['lora_dropout']}, max length {train_info['max_len']} | peak training memory {train_info['peak_train_mem_mb']:.0f} MB | "
    f"{train_info['train_examples']} train / {train_info['val_examples']} validation pairs",
]
table = "\n".join(lines)
print(table)
open(f"{OUT}/efficiency_table.md", "w").write(table)
json.dump({"base": base_stats, "distilled": dist_stats, "merged": ms, "base_disk_mb": base_disk_mb, "adapter_mb": adapter_mb,
           "base_params": base_total_params, "adapter_params": adapter_params, "training": {k: v for k, v in train_info.items() if k != "history"},
           "versions": VERSIONS}, open(f"{OUT}/efficiency.json", "w"), indent=1)

# --- bundle: build the zip OUTSIDE OUT (no self-inclusion), then copy it into OUT ---
bundle_path = f"{OUT}/results_bundle.zip"
if os.path.exists(bundle_path):
    os.remove(bundle_path)                       # leftover from the failed run
local_zip = shutil.make_archive("/content/results_bundle", "zip", OUT)
shutil.copy(local_zip, bundle_path)
print(f"\nbundle: {os.path.getsize(bundle_path) / 2**20:.1f} MB")
print("Saved in", OUT, ":", sorted(os.listdir(OUT)))

GPU: **Tesla T4** | dtype torch.float32 | greedy decoding, max_new_tokens=256, batch size 1 | seed 42

| metric | base | distilled (adapter loaded) | distilled (merged) |
|---|---|---|---|
| parameters (total) | 268,098,176 | 271,895,168 | 268,098,176 |
| trainable parameters (LoRA) | 0 | 3,796,992 | 0 |
| size on disk, full model (MB) | 511 | 511 | 511 |
| adapter size on disk (MB) | - | 14.52 | - |
| peak inference memory (MB) | 1158 | 1181 | 1158 |
| seconds per answer (mean) | 3.99 | 3.35 | 2.15 |
| seconds per answer (median) | 2.02 | 2.18 | 1.69 |
| tokens per second | 25.7 | 21.1 | 32.8 |
| answers that hit 256 tokens | 17 | 1 | 1 |

Training: 25.6 min on Tesla T4 | 3 epochs, 345 steps, effective batch 16, lr 0.0002, LoRA r=16 alpha=32 dropout=0.05, max length 1024 | peak training memory 9626 MB | 1834 train / 100 validation pairs

bundle: 54.4 MB
Saved in /content/drive/MyDrive/amharic_distill_out : ['adapter_epoch1', 'adapter_epoch2', 'adapter_epoch3', 'adapter_final', 'base_o

In [18]:
# 12. Download the files you need (also safe in Google Drive: amharic_distill_out/)
try:
    from google.colab import files
    files.download(PRED_OUT)                       # the file to submit (rename if you wish)
    files.download(f"{OUT}/results_bundle.zip")    # adapter + outputs + tables
except Exception as e:
    print("Download manually from", OUT, "-", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# 12b. Diagnostics: why E01 / E02 / E16 / E17 / E32 / E47 fail (CPU only, uses variables already defined)
import re, numpy as np

TASKS = ["closed_book_knowledge", "instruction_following", "translation", "summarization"]
prompt_of = {r["eval_id"]: r["prompt"] for r in eval_rows}
by_task = {t: [r for r in train_rows if r["task"] == t] for t in TASKS}

def cgrams4(t):
    t = norm(t).replace(" ", "")
    return {t[i:i + 4] for i in range(max(1, len(t) - 3))}
def copy_rate(out, prompt):                       # share of output char-4-grams that also occur in the prompt
    o, p = cgrams4(out), cgrams4(prompt)
    return len(o & p) / len(o) if o else 0.0
def restates(resp, prompt, k=5):                  # answer opens with k words copied from the prompt
    w = norm(resp).split()[:k]
    return len(w) == k and " ".join(w) in norm(prompt)
def repeats(t):                                   # looser than looping(): adjacent duplicate word or repeated bigram
    w = t.split()
    bi = [tuple(w[i:i + 2]) for i in range(len(w) - 1)]
    return any(a == b for a, b in zip(w, w[1:])) or len(bi) != len(set(bi))
def bullet_tail(p): return bool(re.search(r"[•▪]\s*\S", p[-250:]))
def fertility(texts):
    nt = sum(len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts)
    return nt / sum(len(t.split()) for t in texts), nt / sum(len(t) for t in texts)

print("== 1. tokenizer fertility (Gemma-3 tokenizer) ==")
samp = train_rows[:300]
for name, texts in (("train prompts", [r["prompt"] for r in samp]), ("teacher responses", [r["teacher_response"] for r in samp])):
    print("  %-18s %.2f tokens/word, %.2f tokens/char" % ((name,) + fertility(texts)))
print("  %-18s %.2f tokens/word, %.2f tokens/char" % (("English reference",) + fertility(
    ["The question you ask is very important, and everyone should try to answer it carefully."])))

print("\n== 2. scraped '• headline' tails (share of prompts) ==")
for t in TASKS:
    tr = np.mean([bullet_tail(r["prompt"]) for r in by_task[t]])
    ev = np.mean([bullet_tail(r["prompt"]) for r in eval_rows if r["task"] == t])
    print(f"  {t:24s} train {tr:.2f} | eval {ev:.2f}")

def eval_texts(res, t): return [(x["text"], prompt_of[x["eval_id"]]) for x in res if x["task"] == t]

print("\n== 3. copy rate: share of output char-4-grams found in the prompt ==")
for t in TASKS:
    tr = np.mean([copy_rate(r["teacher_response"], r["prompt"]) for r in by_task[t]])
    bs = np.mean([copy_rate(o, p) for o, p in eval_texts(base_results, t)])
    ds = np.mean([copy_rate(o, p) for o, p in eval_texts(dist_results, t)])
    print(f"  {t:24s} teacher {tr:.2f} | base {bs:.2f} | distilled {ds:.2f}")

print("\n== 4. answers that open by restating the prompt (first 5 words) ==")
for t in TASKS:
    tr = np.mean([restates(r["teacher_response"], r["prompt"]) for r in by_task[t]])
    bs = np.mean([restates(o, p) for o, p in eval_texts(base_results, t)])
    ds = np.mean([restates(o, p) for o, p in eval_texts(dist_results, t)])
    print(f"  {t:24s} teacher {tr:.2f} | base {bs:.2f} | distilled {ds:.2f}")

print("\n== 5. repetition: loose flag vs the notebook's looping() ==")
for t in TASKS:
    tr = np.mean([repeats(r["teacher_response"]) for r in by_task[t]])
    ds_loose = np.mean([repeats(o) for o, _ in eval_texts(dist_results, t)])
    ds_strict = np.mean([looping(o) for o, _ in eval_texts(dist_results, t)])
    print(f"  {t:24s} teacher(loose) {tr:.2f} | distilled loose {ds_loose:.2f} | distilled looping() {ds_strict:.2f}")

== 1. tokenizer fertility (Gemma-3 tokenizer) ==
  train prompts      2.93 tokens/word, 0.57 tokens/char
  teacher responses  2.95 tokens/word, 0.55 tokens/char
  English reference  1.13 tokens/word, 0.20 tokens/char

== 2. scraped '• headline' tails (share of prompts) ==
  closed_book_knowledge    train 0.00 | eval 0.00
  instruction_following    train 0.00 | eval 0.00
  translation              train 0.00 | eval 0.00
  summarization            train 0.00 | eval 0.20

== 3. copy rate: share of output char-4-grams found in the prompt ==
  closed_book_knowledge    teacher 0.18 | base 0.52 | distilled 0.44
  instruction_following    teacher 0.90 | base 0.55 | distilled 0.95
  translation              teacher 0.00 | base 0.25 | distilled 0.00
  summarization            teacher 0.56 | base 0.54 | distilled 0.83

== 4. answers that open by restating the prompt (first 5 words) ==
  closed_book_knowledge    teacher 0.40 | base 0.60 | distilled 0.67
  instruction_following    teacher 0.62 | ba

In [20]:
# 13. Checkpoint comparison: epoch 1 vs 2 vs 3 (per-task validation loss + generation diagnostics)
from peft import PeftModel
from sacrebleu.metrics import CHRF
import pandas as pd

N_VAL_GEN = 60            # held-out validation prompts to generate for (chrF++ vs teacher); 0 = skip (saves ~8 min)
CKPTS = {f"epoch_{e}": f"{OUT}/adapter_epoch{e}" for e in (1, 2, 3)}     # epoch_3 == adapter_final
TASKS = ["closed_book_knowledge", "instruction_following", "translation", "summarization"]
chrf = CHRF(word_order=2)
prompt_of = {r["eval_id"]: r["prompt"] for r in eval_rows}

# --- helpers (same definitions as the diagnostics) ---
def cgrams4(t):
    t = norm(t).replace(" ", "")
    return {t[i:i + 4] for i in range(max(1, len(t) - 3))}
def copy_rate(out, prompt):                       # share of output char-4-grams that also occur in the prompt
    o, p = cgrams4(out), cgrams4(prompt)
    return len(o & p) / len(o) if o else 0.0
def loose_repeat(t):                              # adjacent duplicate word or repeated bigram (looser than looping())
    w = t.split()
    bi = [tuple(w[i:i + 2]) for i in range(len(w) - 1)]
    return any(a == b for a, b in zip(w, w[1:])) or len(bi) != len(set(bi))

def eval_metrics(res):
    rows = []
    for t in TASKS:
        xs = [x for x in res if x["task"] == t]
        rows.append({"task": t,
                     "right_script_%": 100 * np.mean([script_ok(x["text"], x["direction"]) for x in xs]),
                     "looping_%": 100 * np.mean([looping(x["text"]) for x in xs]),
                     "repeat_loose_%": 100 * np.mean([loose_repeat(x["text"]) for x in xs]),
                     "copy_rate": np.mean([copy_rate(x["text"], prompt_of[x["eval_id"]]) for x in xs]),
                     "hit_256_%": 100 * np.mean([x["new_tokens"] >= MAX_NEW_TOKENS for x in xs]),
                     "mean_new_tokens": np.mean([x["new_tokens"] for x in xs])})
    return rows

# --- validation pairs aligned with their rows (same length filter as cell 6) ---
val_pairs = []
for r in val_rows:
    ids, lab, _ = enc_pair(r["prompt"], r["teacher_response"])
    if len(ids) <= MAX_LEN: val_pairs.append((r, (ids, lab)))
val_by_task = {t: [p for p in val_pairs if p[0]["task"] == t] for t in TASKS}
print({t: len(v) for t, v in val_by_task.items()}, "validation pairs per task")

store, gen_rows = {}, []
for name, path in CKPTS.items():
    assert os.path.isdir(path), f"missing adapter folder: {path}"
    free_gpu()
    m = PeftModel.from_pretrained(load_base(), path).eval()

    vl = {"val_loss_all": val_loss(m, [p[1] for p in val_pairs])}
    for t in TASKS:
        vl[f"val_loss_{t}"] = val_loss(m, [p[1] for p in val_by_task[t]])

    m = m.merge_and_unload().eval()                       # merged: identical text, faster
    res, _ = run_eval(m, f"{name} (merged)")

    if N_VAL_GEN:                                         # chrF++ against the teacher on held-out validation prompts
        sub = val_pairs[:N_VAL_GEN]
        outs = [generate_one(m, r["prompt"])[0] for r, _ in tqdm(sub, desc=f"{name} val-gen")]
        for t in TASKS:
            idx = [i for i, (r, _) in enumerate(sub) if r["task"] == t]
            vl[f"chrF++_val_{t}"] = (chrf.corpus_score([outs[i] for i in idx],
                                     [[sub[i][0]["teacher_response"] for i in idx]]).score if idx else float("nan"))

    store[name] = {"val": vl, "eval_results": res}
    gen_rows += [{"ckpt": name, **row} for row in eval_metrics(res)]
    del m; free_gpu()

val_df = pd.DataFrame({k: v["val"] for k, v in store.items()}).T.round(3)
gen_df = pd.DataFrame(gen_rows).round(2)
print("\n=== validation loss per task (+ chrF++ vs teacher on held-out validation prompts) ===")
display(val_df)
print("\n=== generation diagnostics on the 60 eval prompts ===")
display(gen_df)

val_df.to_csv(f"{OUT}/checkpoint_val_metrics.csv")
gen_df.to_csv(f"{OUT}/checkpoint_gen_metrics.csv", index=False)
json.dump(store, open(f"{OUT}/checkpoint_comparison.json", "w"), ensure_ascii=False, indent=1)
print("saved ->", OUT)

{'closed_book_knowledge': 29, 'instruction_following': 18, 'translation': 26, 'summarization': 27} validation pairs per task


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

epoch_1 (merged):   0%|          | 0/60 [00:00<?, ?it/s]

{
 "label": "epoch_1 (merged)",
 "sec_per_answer_mean": 3.374179235316721,
 "sec_per_answer_median": 2.7143012720002844,
 "tokens_per_sec": 28.456302991164,
 "total_new_tokens": 5761,
 "peak_mem_mb": 2178.76025390625,
 "n_hit_256": 9
}


epoch_1 val-gen:   0%|          | 0/60 [00:00<?, ?it/s]

...


=== validation loss per task (+ chrF++ vs teacher on held-out validation prompts) ===


,val_loss_all,val_loss_closed_book_knowledge,val_loss_instruction_following,val_loss_translation,val_loss_summarization,chrF++_val_closed_book_knowledge,chrF++_val_instruction_following,chrF++_val_translation,chrF++_val_summarization
epoch_1,1.631,2.350,0.384,2.666,1.156,22.037,48.288,15.412,35.917
epoch_2,1.518,2.189,0.363,2.539,1.060,22.091,44.104,22.220,34.416
epoch_3,1.506,2.165,0.354,2.583,1.047,22.935,42.106,23.145,31.875



=== generation diagnostics on the 60 eval prompts ===


,ckpt,task,right_script_%,looping_%,repeat_loose_%,copy_rate,hit_256_%,mean_new_tokens
0,epoch_1,closed_book_knowledge,80.00,33.33,80.00,0.55,33.33,137.80
1,epoch_1,instruction_following,100.00,0.00,20.00,0.97,0.00,47.67
2,epoch_1,translation,93.33,20.00,53.33,0.12,20.00,69.00
3,epoch_1,summarization,100.00,0.00,33.33,0.88,6.67,129.60
4,epoch_2,closed_book_knowledge,80.00,40.00,80.00,0.56,33.33,134.20
5,epoch_2,instruction_following,100.00,0.00,20.00,0.97,0.00,43.33
6,epoch_2,translation,100.00,6.67,20.00,0.04,6.67,38.07
7,epoch_2,summarization,100.00,0.00,6.67,0.85,0.00,110.33
8,epoch_3,closed_book_knowledge,93.33,20.00,73.33,0.44,6.67,101.27
9,epoch_3,instruction_following,100.00,0.00,6.67,0.95,0.00,44.33


saved -> /content/drive/MyDrive/amharic_distill_out
